# Optimize clustered synthetic data parameters

This notebook searches for a clustered synthetic data-generating process such that:

- the **IID** engine declares equivalence at the chosen margin, and
- the **cluster-aware** engine does **not** declare equivalence.

All synthetic-data optimization notebooks in this repository follow the same flow:
environment setup, configuration, search, summary, save, validation, and diagnostics.


## 1. Environment setup


In [ ]:
from pyTOST.data_gen.notebook_utils import (
    configure_notebook_environment,
    summarize_result,
    save_json,
    result_payload,
)

REPO_ROOT = configure_notebook_environment()
ALPHA = 0.05
MARGIN = 1.0
SEARCH_SEED = 123
VALIDATION_SEED = 123
OUTPUT_JSON = "best_params_cluster.json"

ENGINE_SPECS = {
    "IID": {"ci": "ci_iid", "eq": "eq_iid", "mu": "mu_iid"},
    "Cluster": {"ci": "ci_cluster", "eq": "eq_cluster", "mu": "mu_cluster"},
}


## 2. Configure the optimization problem


In [ ]:
from pyTOST.data_gen.optimize_cluster_params import Bounds, optimize, evaluate

bounds = Bounds()
MAXITER = 80
POPSIZE = 16
POLISH = False
WORKERS = 1


## 3. Run the optimization


In [ ]:
opt_result = optimize(
    alpha=ALPHA,
    margin=MARGIN,
    seed=SEARCH_SEED,
    bounds=bounds,
    maxiter=MAXITER,
    popsize=POPSIZE,
    polish=POLISH,
    rng_seed=SEARCH_SEED,
    workers=WORKERS,
    verbose=True,
)

best_params = opt_result["best_kwargs"]
best_diag = opt_result["diagnostics"]
best_summary = summarize_result(best_diag, ENGINE_SPECS)
best_summary


## 4. Summarize the best candidate


In [ ]:
best_payload = result_payload(
    params=best_params,
    summary_source=best_diag,
    engine_specs=ENGINE_SPECS,
    extra={
        "optimizer_success": best_diag.get("optimizer_success"),
        "optimizer_message": best_diag.get("optimizer_message"),
        "nit": best_diag.get("nit"),
        "nfev": best_diag.get("nfev"),
    },
)
best_payload


## 5. Save the best parameters


In [ ]:
save_json(best_payload, OUTPUT_JSON)
OUTPUT_JSON


## 6. Validate by regenerating data and rerunning the target engines


In [ ]:
validation = evaluate(best_params, alpha=ALPHA, margin=MARGIN, seed=VALIDATION_SEED)
validation_summary = summarize_result(validation, ENGINE_SPECS)
validation_summary


## 7. Search diagnostics


In [ ]:
import pandas as pd

diagnostics_df = pd.DataFrame([best_diag])
diagnostics_df
